In [3]:
import sys
import os
import json
import copy
import openpyxl
import numpy as np
import matplotlib.pyplot as plt

In [4]:
BACKEND_PATH = os.path.abspath(os.path.join(os.getcwd(), 'backend'))
sys.path.insert(0, BACKEND_PATH)

CONFIG_FILE  = os.path.join(BACKEND_PATH, 'config.json')
FORMULAS_PATH = os.path.join(BACKEND_PATH, 'formulas_map.json')

from excel_formula_engine import ExcelFormulaProcessor
processor = ExcelFormulaProcessor()
print("Backend:", BACKEND_PATH)

Backend: c:\Users\agarr\OneDrive\Escritorio\IIT\CC-WBT-AI\backend


In [5]:
# User configuration
COUNTRY = 'Rwanda'
MODEL   = 'CleanStep'

# Load from config
with open(CONFIG_FILE) as f:
    config = json.load(f)

FUELS = config['FUELS'][COUNTRY]['expanded']
EXPECTED_SHEETS = config['FUELS'][COUNTRY]['more_expanded']  
YEAR_RANGE = config['COUNTRY_YEAR_RANGES'][COUNTRY]
YEARS = list(range(YEAR_RANGE['start'], YEAR_RANGE['end'] + 1))

# Dynamic paths
SCENARIO_PATH = os.path.join(BACKEND_PATH, COUNTRY)
DESIGN_CAPITAL_PATH = os.path.join(SCENARIO_PATH, f'design-capital-{MODEL}.xlsx')
FINANCIAL_STMT_PATH = os.path.join(SCENARIO_PATH, f'financial-statements-{MODEL}.xlsx')

print(f"Country: {COUNTRY} | Model: {MODEL}")
print(f"Years: {YEARS[0]}–{YEARS[-1]}")
print(f"Fuels: {FUELS}")
print(f"Expected sheets: {EXPECTED_SHEETS}")

Country: Rwanda | Model: CleanStep
Years: 2023–2034
Fuels: ['Electricity & E-Cooking', 'Electricity (Low access)', 'LPG']
Expected sheets: ['Electricity (Only E-Cooking)', 'Electricity & E-Cooking', 'Electricity (Low access)', 'LPG']


In [11]:
# Helper functions
def read_capital_params(sheet_name):
    wb = openpyxl.load_workbook(DESIGN_CAPITAL_PATH, data_only=True)
    ws = wb[sheet_name]
    params = {}
    for row in ws.iter_rows(max_col=3, values_only=True):
        if row[0] in ['1. Equity', 'Cost of Equity', '2. Grants', 'Years realisation', 'Cost of Debt', 'Grace period', 'Amortization period']:
            params[row[0]] = row[2]
    wb.close()
    return params

def write_capital_param(sheet_name, param_name, new_value):
    wb = openpyxl.load_workbook(DESIGN_CAPITAL_PATH)
    ws = wb[sheet_name]
    for row in ws.iter_rows(max_col=3):
        if row[0].value == param_name:
            row[2].value = new_value
            break
    wb.save(DESIGN_CAPITAL_PATH)
    wb.close()

def run_engine():
    design_rel = os.path.join(COUNTRY, f'design-capital-{MODEL}.xlsx')
    financial_rel = os.path.join(COUNTRY, f'financial-statements-{MODEL}.xlsx')
    original_dir = os.getcwd()
    os.chdir(BACKEND_PATH)
    
    try:
        processor.clear_workbook_cache()
        processor.apply_formulas(
            file_path=design_rel,
            formulas_json_path='formulas_map.json',
            country=COUNTRY,
            models=[MODEL],
            fuels=FUELS,
            expected_sheets=EXPECTED_SHEETS
        )
        processor.clear_workbook_cache()
        processor.apply_formulas(
            file_path=financial_rel,
            formulas_json_path='formulas_map.json',
            country=COUNTRY,
            models=[MODEL],
            fuels=FUELS,
            expected_sheets=EXPECTED_SHEETS
        )
    finally:
        os.chdir(original_dir)

def read_outputs(sheet_name):
    wb = openpyxl.load_workbook(FINANCIAL_STMT_PATH, data_only=True)
    ws = wb[sheet_name]
    rows_of_interest = ['Long term subsidies', 'Operating Cash Flow', 'Financial Expense', 'Debt repayment']
    outputs = {}
    for row in ws.iter_rows(values_only=True):
        if row[0] in rows_of_interest and row[1] == '-':
            outputs[row[0]] = [v if v is not None else 0 for v in row[3:3+len(YEARS)]]
    wb.close()
    return outputs

def compute_dscr(outputs, ex_subsidies=True):
    ocf     = outputs['Operating Cash Flow']
    fin_exp = outputs['Financial Expense']
    debt_rep = outputs['Debt repayment']
    lts     = outputs['Long term subsidies']

    cf = [ocf[i] - lts[i] for i in range(len(YEARS))] if ex_subsidies else ocf

    result = []
    for i in range(len(YEARS)):
        debt_service = abs(fin_exp[i]) + abs(debt_rep[i])
        result.append(round(cf[i] / debt_service, 2) if debt_service > 0 else None)
    return result

In [12]:
SHEET = 'Electricity & E-Cooking'
PARAM = 'Cost of Equity'
ORIGINAL_VALUE = read_capital_params(SHEET)[PARAM]

values_to_test = list(range(10, 22, 2))  # 10, 12, 14, 16, 18, 20
results = []

for val in values_to_test:
    write_capital_param(SHEET, PARAM, val)
    run_engine()
    out = read_outputs(SHEET)
    lts_total = round(sum(out['Long term subsidies']), 1)
    dscr = compute_dscr(out)
    dscr_min = min([d for d in dscr if d is not None])
    results.append({'value': val, 'lts_total': lts_total, 'dscr_min': dscr_min})
    print(f"  {PARAM} = {val}% → LTS total: {lts_total} M$ | DSCR min: {dscr_min}")

# Restore original value
write_capital_param(SHEET, PARAM, ORIGINAL_VALUE)
run_engine()
print(f"\nRestored {PARAM} to {ORIGINAL_VALUE}%")

  Cost of Equity = 10% → LTS total: 368.3 M$ | DSCR min: 7.81
  Cost of Equity = 12% → LTS total: 827.8 M$ | DSCR min: 4.47
  Cost of Equity = 14% → LTS total: 1295.0 M$ | DSCR min: 4.47
  Cost of Equity = 16% → LTS total: 1774.5 M$ | DSCR min: 4.47
  Cost of Equity = 18% → LTS total: 2253.9 M$ | DSCR min: 4.47
  Cost of Equity = 20% → LTS total: 2746.2 M$ | DSCR min: 4.47

Restored Cost of Equity to 16%


In [13]:
# Diagnose DSCR year by year at baseline (Cost of Equity = 16%)
SHEET = 'Electricity & E-Cooking'
out = read_outputs(SHEET)

ocf      = out['Operating Cash Flow']
fin_exp  = out['Financial Expense']
debt_rep = out['Debt repayment']
lts      = out['Long term subsidies']

print(f"{'Year':<6} {'OCF':>8} {'LTS':>8} {'OCF-LTS':>8} {'FinExp':>8} {'DebtRep':>8} {'DebtSvc':>8} {'DSCR':>8}")
print("-" * 65)
for i, y in enumerate(YEARS):
    debt_svc = abs(fin_exp[i]) + abs(debt_rep[i])
    ocf_ex   = ocf[i] - lts[i]
    dscr_val = round(ocf_ex / debt_svc, 2) if debt_svc > 0 else None
    print(f"{y:<6} {ocf[i]:>8.1f} {lts[i]:>8.1f} {ocf_ex:>8.1f} {fin_exp[i]:>8.1f} {debt_rep[i]:>8.1f} {debt_svc:>8.1f} {str(dscr_val):>8}")

Year        OCF      LTS  OCF-LTS   FinExp  DebtRep  DebtSvc     DSCR
-----------------------------------------------------------------
2023      102.2      0.0    102.2      0.0      0.0      0.0     None
2024      148.7      0.0    148.7      0.0      0.0      0.0     None
2025      227.9     27.0    200.8      0.0      0.0      0.0     None
2026      303.9     78.1    225.9      0.0      0.0      0.0     None
2027      377.0    115.4    261.5      0.0      0.0      0.0     None
2028      447.0    147.5    299.5     -4.8      0.0      4.8    62.39
2029      522.9    182.8    340.0    -16.0      0.0     16.0    21.25
2030      604.3    210.4    393.8    -29.2      0.0     29.2    13.49
2031      682.3    232.6    449.7    -41.6      0.0     41.6    10.81
2032      756.9    253.7    503.2    -51.0      0.0     51.0     9.87
2033      828.2    274.2    554.0    -57.2      0.0     57.2     9.68
2034      867.8    252.7    615.2    -58.4    -29.8     88.2     6.97
